# Stage 5 — ML Risk Engine

**Medical Device Safety Event Severity Classification**  
All metrics loaded from pre-computed experiment artifacts. No re-training in this notebook.

## 0. Imports & Configuration

In [ ]:
import json
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from sklearn.metrics import precision_recall_curve, roc_curve, auc

# ── Path setup ──
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
FEATURES_DIR = ROOT / "data" / "features"
EXPERIMENTS_DIR = ROOT / "models" / "experiments"
PRODUCTION_DIR = ROOT / "models" / "production"
METADATA_COLS = {"id", "device_id", "manufacturer_id", "event_date", "event_date_available"}
TARGET_COL = "is_class_i"

# ── Plotting style ──
plt.rcParams.update({
    "figure.facecolor": "#0F1117",
    "axes.facecolor": "#1A1D27",
    "axes.edgecolor": "#2E3148",
    "axes.labelcolor": "#C9D1D9",
    "xtick.color": "#8B949E",
    "ytick.color": "#8B949E",
    "text.color": "#C9D1D9",
    "grid.color": "#2E3148",
    "grid.linestyle": "--",
    "grid.alpha": 0.6,
    "axes.titlecolor": "#F0F6FF",
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.family": "DejaVu Sans",
    "lines.linewidth": 2.0,
    "legend.framealpha": 0.15,
    "legend.edgecolor": "#3E4568",
})
PALETTE = ["#58A6FF", "#3FB950", "#F78166", "#D2A8FF"]

def load_split(name):
    df = pd.read_parquet(FEATURES_DIR / f"{name}.parquet")
    feat_cols = [c for c in df.columns if c not in METADATA_COLS and c != TARGET_COL]
    X = df[feat_cols].values.astype(np.float32)
    y = df[TARGET_COL].values.astype(int)
    return X, y, feat_cols

print("Configuration OK.")
print(f"  Features dir : {FEATURES_DIR}")
print(f"  Experiments  : {EXPERIMENTS_DIR}")
print(f"  Production   : {PRODUCTION_DIR}")


## 1. Data Dimensions

In [ ]:
splits = {}
for name in ["train", "validation", "test", "holdout_2018"]:
    X, y, fcols = load_split(name)
    splits[name] = {"X": X, "y": y, "feature_cols": fcols}

print(f"{'Split':<15} {'Rows':>8} {'Features':>10} {'Positives':>10} {'Pos Rate':>10}")
print("-" * 60)
for name, s in splits.items():
    pos = s['y'].sum()
    total = len(s['y'])
    print(f"{name:<15} {total:>8,} {len(s['feature_cols']):>10} {pos:>10,} {pos/total*100:>9.2f}%")


## 2. Validation Leaderboard

In [ ]:
experiments = []
for p in sorted(EXPERIMENTS_DIR.rglob("val_metrics.json")):
    meta = json.loads(p.read_text())
    experiments.append({
        "name": meta["model_name"],
        "exp_dir": p.parent,
        "val_pr_auc": meta["val_metrics"]["pr_auc"],
        "val_roc_auc": meta["val_metrics"]["roc_auc"],
        "val_f1": meta["val_metrics"]["f1"],
        "val_recall": meta["val_metrics"]["recall"],
        "val_precision": meta["val_metrics"]["precision"],
        "meta": meta,
    })
experiments.sort(key=lambda e: e["val_pr_auc"], reverse=True)

df_lb = pd.DataFrame([{
    "Model": e["name"],
    "PR-AUC": f"{e['val_pr_auc']:.4f}",
    "ROC-AUC": f"{e['val_roc_auc']:.4f}",
    "Recall": f"{e['val_recall']:.4f}",
    "Precision": f"{e['val_precision']:.4f}",
    "F1": f"{e['val_f1']:.4f}",
} for e in experiments])
print("Validation Leaderboard (sorted by PR-AUC):")
print(df_lb.to_string(index=False))
print(f"\n>>> WINNER: {experiments[0]['name']} (PR-AUC = {experiments[0]['val_pr_auc']:.4f})")


## 3. Precision-Recall Curves — All Models (Validation)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.set_facecolor("#1A1D27")

X_val, y_val, _ = splits["validation"]["X"], splits["validation"]["y"], None
baseline_pr = y_val.mean()
ax.axhline(baseline_pr, color="#8B949E", linestyle="--", linewidth=1.2, label=f"Random baseline (PR={baseline_pr:.3f})")

for i, exp in enumerate(experiments):
    npz = np.load(exp["exp_dir"] / "val_predictions.npz")
    y_proba = npz["y_proba"]
    prec, rec, _ = precision_recall_curve(y_val, y_proba)
    pr_auc = auc(rec, prec)
    lw = 2.8 if i == 0 else 1.6
    ax.plot(rec, prec, color=PALETTE[i % len(PALETTE)], linewidth=lw,
            label=f"{exp['name']}  (PR-AUC={pr_auc:.4f})")

ax.set_xlabel("Recall", fontsize=11)
ax.set_ylabel("Precision", fontsize=11)
ax.set_title("Precision–Recall Curves (Validation 2015)", fontsize=13)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
plt.savefig(ROOT / "models" / "production" / "pr_curves_validation.png", dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()
print("Saved: models/production/pr_curves_validation.png")


## 4. ROC Curves — All Models (Validation)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot([0, 1], [0, 1], color="#8B949E", linestyle="--", linewidth=1.2, label="Random (AUC=0.50)")

for i, exp in enumerate(experiments):
    npz = np.load(exp["exp_dir"] / "val_predictions.npz")
    y_proba = npz["y_proba"]
    fpr, tpr, _ = roc_curve(y_val, y_proba)
    roc_auc = auc(fpr, tpr)
    lw = 2.8 if i == 0 else 1.6
    ax.plot(fpr, tpr, color=PALETTE[i % len(PALETTE)], linewidth=lw,
            label=f"{exp['name']}  (AUC={roc_auc:.4f})")

ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate", fontsize=11)
ax.set_title("ROC Curves (Validation 2015)", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
fig.tight_layout()
plt.savefig(ROOT / "models" / "production" / "roc_curves_validation.png", dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()
print("Saved: models/production/roc_curves_validation.png")


## 5. Best Model: Test-Set Confusion Matrix

In [ ]:
import matplotlib.colors as mcolors

tm = json.loads((PRODUCTION_DIR / "test_metrics.json").read_text())
test_m = tm["test"]
threshold = tm["decision_threshold"]

best_model = joblib.load(PRODUCTION_DIR / "model.pkl")
X_test, y_test, feat_cols = splits["test"]["X"], splits["test"]["y"], splits["test"]["feature_cols"]
y_proba_test = best_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= threshold).astype(int)

cm = np.array([[test_m["tn"], test_m["fp"]],
               [test_m["fn"], test_m["tp"]]])
labels = [["TN", "FP"], ["FN", "TP"]]

fig, ax = plt.subplots(figsize=(6, 5))
colors = np.array([[0.4, 0.1], [0.1, 0.9]])
cmap = plt.cm.RdYlGn
im = ax.imshow(colors, cmap=cmap, vmin=0, vmax=1, aspect="auto")
for i in range(2):
    for j in range(2):
        val = cm[i, j]
        lbl = labels[i][j]
        ax.text(j, i, f"{lbl}\n{val:,}", ha="center", va="center", fontsize=14,
                color="white", fontweight="bold")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Predicted\nNegative", "Predicted\nPositive"], fontsize=10)
ax.set_yticklabels(["Actual\nNegative", "Actual\nPositive"], fontsize=10)
ax.set_title(f"Confusion Matrix — Test Set 2016–2017\n(threshold={threshold:.4f})", fontsize=12)
for spine in ax.spines.values():
    spine.set_edgecolor("#2E3148")
fig.tight_layout()
plt.savefig(PRODUCTION_DIR / "confusion_matrix_test.png", dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()
print(f"Test PR-AUC: {test_m['pr_auc']:.4f}  ROC-AUC: {test_m['roc_auc']:.4f}")
print(f"Precision: {test_m['precision']:.4f}  Recall: {test_m['recall']:.4f}  F1: {test_m['f1']:.4f}")


## 6. Top-20 Feature Importances

In [ ]:
fi = json.loads((PRODUCTION_DIR / "feature_importance.json").read_text())
top_n = 20
df_fi = pd.DataFrame(fi[:top_n])

fig, ax = plt.subplots(figsize=(10, 7))
colors_bar = [PALETTE[0]] * 3 + [PALETTE[3]] + [PALETTE[1]] * 2 + [PALETTE[2]] * 3 + ["#8B949E"] * 10
bars = ax.barh(
    df_fi["feature"][::-1],
    df_fi["importance"][::-1],
    color=colors_bar[::-1],
    edgecolor="#2E3148",
    linewidth=0.5,
    height=0.7,
)
for bar, val in zip(bars, df_fi["importance"][::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", ha="left", fontsize=7.5, color="#8B949E")

ax.set_xlabel("Mean Decrease in Impurity (MDI)", fontsize=11)
ax.set_title("Top-20 Feature Importances — Random Forest", fontsize=13)
ax.grid(True, axis="x")
ax.set_xlim(0, df_fi["importance"].max() * 1.18)

legend_handles = [
    mpatches.Patch(color=PALETTE[0], label="Category history"),
    mpatches.Patch(color=PALETTE[3], label="Manufacturer freq."),
    mpatches.Patch(color=PALETTE[1], label="Text length"),
    mpatches.Patch(color=PALETTE[2], label="Manufacturer history"),
    mpatches.Patch(color="#8B949E", label="Other"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8)
fig.tight_layout()
plt.savefig(PRODUCTION_DIR / "feature_importance_top20.png", dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()


## 7. Cross-Split Metrics Summary

In [ ]:
mc = json.loads((PRODUCTION_DIR / "model_card.json").read_text())
metrics_data = [
    {"Split": "Validation 2015", **mc["metrics"]["validation"]},
    {"Split": "Test 2016-17", **mc["metrics"]["test"]},
    {"Split": "Holdout 2018", **mc["metrics"]["holdout_2018"]},
]
df_summary = pd.DataFrame(metrics_data)[["Split", "pr_auc", "roc_auc", "f1", "precision", "recall"]]
df_summary.columns = ["Split", "PR-AUC", "ROC-AUC", "F1", "Precision", "Recall"]
print(df_summary.to_string(index=False))

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
metrics_to_plot = ["PR-AUC", "ROC-AUC", "F1"]
split_labels = ["Val 2015", "Test 2016-17", "Holdout 2018"]
colors_split = [PALETTE[0], PALETTE[1], PALETTE[3]]

for ax_idx, metric in enumerate(metrics_to_plot):
    vals = df_summary[metric].tolist()
    ax = axes[ax_idx]
    bars = ax.bar(split_labels, vals, color=colors_split, edgecolor="#2E3148", linewidth=0.5, width=0.55)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
                f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1.0)
    ax.grid(True, axis="y")
    ax.set_facecolor("#1A1D27")
    for spine in ax.spines.values():
        spine.set_edgecolor("#2E3148")

fig.suptitle("Random Forest — Metrics Across Temporal Splits", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.savefig(PRODUCTION_DIR / "cross_split_metrics.png", dpi=150, bbox_inches="tight", facecolor="#0F1117")
plt.show()


## 8. Summary

**Best Model: Random Forest** (300 trees, `class_weight='balanced'`)

| Split | PR-AUC | ROC-AUC | F1 | Precision | Recall |
|-------|--------|---------|-----|-----------|--------|
| Validation 2015 | 0.6055 | 0.8880 | 0.374 | 0.248 | 0.760 |
| **Test 2016–2017** | **0.5420** | **0.8611** | **0.504** | **0.977** | **0.339** |
| Holdout 2018 | 0.6869 | 0.8724 | 0.655 | 0.949 | 0.500 |

Decision threshold: **0.8555** (tuned on validation to maximise F1)

Production artifacts written to `models/production/`.